In [1]:

%pip install svgwrite --upgrade --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:

%pprint
import sys
import os.path as osp, os as os

executable_path = sys.executable; scripts_folder = osp.join(osp.dirname(executable_path), 'Scripts')
if (scripts_folder not in sys.path): sys.path.insert(1, scripts_folder)
py_folder = osp.abspath(osp.join(os.pardir, 'py'))
if (py_folder not in sys.path): sys.path.insert(1, py_folder)
ffmpeg_folder = r'C:\ProgramData\chocolatey\lib\ffmpeg-full\tools\ffmpeg\bin'
if (ffmpeg_folder not in sys.path): sys.path.insert(1, ffmpeg_folder)
scripts_folder = r'C:\Users\daveb\AppData\Roaming\Python\Python312\Scripts'
if (scripts_folder not in sys.path): sys.path.insert(1, scripts_folder)
import math
import svgwrite
from cairosvg import svg2pdf
import subprocess

Pretty printing has been turned OFF


In [3]:

# SVG parameters
svg_width = 683.18048
svg_height = 683.18048

# Center and radii from SVG
cx, cy = 361.24728, 344.75668
marker_radius = 325
r_outer = 292.18814
num_spokes = 24

# 24-hour labels
labels = ["24"] + [str(i) for i in range(1, 24)]

In [4]:

defs_style = """
        .line-style {
            fill: none;
            stroke-dasharray: none;
            stroke-opacity: 1;
            stroke-width: 1.0;
            stroke: #000000;
        }
        .label-style {
            -inkscape-font-specification: 'Arial, Normal';
            fill-opacity: 1;
            fill: #000000;
            font-family: Arial;
            font-size: 24px;
            font-stretch: normal;
            font-style: normal;
            font-variant-caps: normal;
            font-variant-east-asian: normal;
            font-variant-ligatures: none;
            font-variant-numeric: normal;
            font-variant: normal;
            font-weight: normal;
            line-height: 1.25;
            stroke-dasharray: none;
            stroke-opacity: 1;
            stroke-width: 1.33333;
            stroke: none;
            text-align: center;
            text-decoration-color: #000000;
        }
        .marker-text-style {
            -inkscape-font-specification: 'Century Schoolbook, Normal';
            fill-opacity: 1;
            fill: #000000;
            font-family: 'Century Schoolbook';
            font-size: 36px;
            font-stretch: normal;
            font-style: normal;
            font-variant-caps: normal;
            font-variant-east-asian: normal;
            font-variant-ligatures: none;
            font-variant-numeric: normal;
            font-variant: normal;
            font-weight: normal;
            line-height: 1.25;
            stroke-opacity: 1;
            stroke-width: 1.9647;
            stroke: none;
            text-align: center;
            text-anchor: middle;
            text-decoration-color: #000000;
            white-space: pre;
        }
        .marker-tspan-style {
            -inkscape-font-specification:'Century Schoolbook, Normal';
            font-family:'Century Schoolbook';
            font-size:36px;
            font-stretch:normal;
            font-style:normal;
            font-variant-caps:normal;
            font-variant-east-asian:normal;
            font-variant-ligatures:none;
            font-variant-numeric:normal;
            font-variant:normal;
            font-weight:normal;
            stroke-width:1.9647;
        }"""
named_view = '''
  <sodipodi:namedview
     id="namedview1"
     pagecolor="#ffffff"
     bordercolor="#000000"
     borderopacity="0.25"
     inkscape:showpageshadow="2"
     inkscape:pageopacity="0.0"
     inkscape:pagecheckerboard="0"
     inkscape:deskcolor="#d1d1d1"
     showgrid="false"
     inkscape:zoom="1.0196947"
     inkscape:cx="330.00073"
     inkscape:cy="330.49108"
     inkscape:window-width="1600"
     inkscape:window-height="912"
     inkscape:window-x="-8"
     inkscape:window-y="1692"
     inkscape:window-maximized="1"
     inkscape:current-layer="svg1" />
  '''


----

In [5]:

diagrams_folder = r'C:\Users\daveb\OneDrive\Documents\Projects\Diagrams'
file_prefix = 'clock_spokes_and_labels'
svg_path = osp.join(diagrams_folder, f'{file_prefix}.svg')
dwg = svgwrite.Drawing(
    svg_path,
    profile='full',
    size=(svg_width, svg_height)
)
dwg.viewbox(-5, -5, svg_width+40, svg_height+40)

In [6]:

# Add style block
style_container = dwg.defs.add(dwg.style(defs_style))

In [7]:

# Add group for drawing
group_drawing = dwg.g(id="group-drawing")

In [8]:

# Outer ellipse (clock face)
ellipse_shape = group_drawing.add(
    dwg.ellipse(
        center=(cx, cy),
        r=(r_outer, r_outer),
        class_="line-style",
        id="path-outer-circle"
    )
)

In [9]:

# Inner dot
ellipse_shape = group_drawing.add(
    dwg.ellipse(
        id="circle-center",
        center=(cx, cy),
        r=(10, 10),
        style="fill:#000000;fill-opacity:1;stroke:none"
    )
)

In [10]:

# Add group for markers
group_markers = dwg.g(id="group-markers")

for i, label in enumerate(labels):
    # Angle for each marker (0 at top, increases clockwise)
    angle_deg = (i * 360 / 24) - 90
    angle_rad = math.radians(angle_deg)
    x = cx + marker_radius * math.cos(angle_rad)
    y = cy + marker_radius * math.sin(angle_rad)
    # Rotate so text is tangential to the circle (upright at top, rotated at sides)
    # For outward-facing baseline, add 90 degrees to angle
    rotate_angle = angle_deg + 90
    # Compose transform string
    transform = f"rotate({rotate_angle:.2f},{x:.2f},{y:.2f})"
    # Create text element
    text = dwg.text(
        label,
        insert=(x, y),
        class_="marker-text-style",
        id=f"text-{label}",
        transform=transform
    )
    # Add tspan for structure
    text.add(dwg.tspan(
        label,
        x=[x],
        y=[y],
        class_="marker-tspan-style",
        id=f"tspan{label}"
    ))
    group_markers.add(text)

group_container = group_drawing.add(group_markers)

In [11]:

# Add group for spokes
group_spokes = dwg.g(id="group-spokes", **{"style": "display:inline"})

# Generate spokes
for i in range(num_spokes):
    angle = math.radians((360 / num_spokes) * i - 90)  # start at 12 o'clock
    x_outer = cx + r_outer * math.cos(angle)
    y_outer = cy + r_outer * math.sin(angle)

    # Path command
    path_d = f'M {x_outer:.5f},{y_outer:.5f} {cx:.5f},{cy:.5f}'
    group_spokes.add(
        dwg.path(
            d=path_d,
            class_="line-style",
            id=f'path-{i:02d}-30'
        )
    )

group_container = group_drawing.add(group_spokes)

In [12]:

dwg.add(group_drawing)
dwg.save()


----

In [13]:

# Convert the SVG to PDF
%pip install cairosvg --upgrade --quiet

pdf_path = osp.join(diagrams_folder, f'{file_prefix}.pdf')
svg2pdf(
    url=svg_path, write_to=pdf_path, output_width=225, output_height=225
)

# Open in Microsoft Edge
edge_path = r"C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe"
Popen_obj = subprocess.Popen([edge_path, '--profile-directory=Default', pdf_path])

Note: you may need to restart the kernel to use updated packages.


In [14]:

# Post-process the SVG file as text and insert the Inkscape-specific metadata
with open(svg_path, 'r', encoding='utf-8') as f:
    svg_content = f.read()

# Insert before the <defs
svg_with_namedview = svg_content.replace('<defs', named_view + '<defs')

with open(svg_path, 'w', encoding='utf-8') as f:
    f.write(svg_with_namedview)

# Open in Inkscape
inkscape_path = r'C:\Program Files\Inkscape\bin\inkscape.exe'
Popen_obj = subprocess.Popen([inkscape_path, svg_path])